In [1]:
!pip install pandas numpy scikit-learn tensorflow

INFO: pip is looking at multiple versions of ml-dtypes to determine which version is compatible with other requirements. This could take a while.
   ---------------------------------------- 0.0/350.8 MB ? eta -:--:--
   ---------------------------------------- 0.2/350.8 MB 3.5 MB/s eta 0:01:41
   ---------------------------------------- 0.5/350.8 MB 5.2 MB/s eta 0:01:07
   ---------------------------------------- 0.9/350.8 MB 6.8 MB/s eta 0:00:52
   ---------------------------------------- 1.1/350.8 MB 5.7 MB/s eta 0:01:01
   ---------------------------------------- 1.5/350.8 MB 6.3 MB/s eta 0:00:56
   ---------------------------------------- 1.9/350.8 MB 6.6 MB/s eta 0:00:53
   ---------------------------------------- 2.2/350.8 MB 6.9 MB/s eta 0:00:51
   ---------------------------------------- 2.4/350.8 MB 6.3 MB/s eta 0:00:56
   ---------------------------------------- 2.7/350.8 MB 6.4 MB/s eta 0:00:55
   ---------------------------------------- 3.0/350.8 MB 6.4 MB/s eta 0:00:55
   

In [2]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import accuracy_score, confusion_matrix

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

In [5]:
df = pd.read_csv("Churn_Modelling.csv")

print(df.head())
print(df.shape)

   RowNumber  CustomerId   Surname  CreditScore Geography  Gender  Age  \
0          1    15634602  Hargrave          619    France  Female   42   
1          2    15647311      Hill          608     Spain  Female   41   
2          3    15619304      Onio          502    France  Female   42   
3          4    15701354      Boni          699    France  Female   39   
4          5    15737888  Mitchell          850     Spain  Female   43   

   Tenure    Balance  NumOfProducts  HasCrCard  IsActiveMember  \
0       2       0.00              1          1               1   
1       1   83807.86              1          0               1   
2       8  159660.80              3          1               0   
3       1       0.00              2          0               0   
4       2  125510.82              1          1               1   

   EstimatedSalary  Exited  
0        101348.88       1  
1        112542.58       0  
2        113931.57       1  
3         93826.63       0  
4         790

In [7]:
df = df.drop(['RowNumber', 'CustomerId', 'Surname'], axis=1)

df['Gender'] = LabelEncoder().fit_transform(df['Gender'])

df = pd.get_dummies(df, columns=['Geography'], drop_first=True)

print(df.head())

   CreditScore  Gender  Age  Tenure    Balance  NumOfProducts  HasCrCard  \
0          619       0   42       2       0.00              1          1   
1          608       0   41       1   83807.86              1          0   
2          502       0   42       8  159660.80              3          1   
3          699       0   39       1       0.00              2          0   
4          850       0   43       2  125510.82              1          1   

   IsActiveMember  EstimatedSalary  Exited  Geography_Germany  Geography_Spain  
0               1        101348.88       1              False            False  
1               1        112542.58       0              False             True  
2               0        113931.57       1              False            False  
3               0         93826.63       0              False            False  
4               1         79084.10       0              False             True  


In [9]:
X = df.drop('Exited', axis=1)
y = df['Exited']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [11]:
sc = StandardScaler()

X_train = sc.fit_transform(X_train)
X_test = sc.transform(X_test)

In [13]:
model = Sequential()

model.add(Dense(16, activation='relu', input_shape=(X_train.shape[1],)))
model.add(Dense(8, activation='relu'))
model.add(Dense(1, activation='sigmoid'))

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.fit(X_train, y_train, epochs=10, batch_size=32)

Epoch 1/10


C:\Users\m3112\anaconda3\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 828us/step - accuracy: 0.7816 - loss: 0.5205
Epoch 2/10
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 787us/step - accuracy: 0.7945 - loss: 0.4401
Epoch 3/10
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 820us/step - accuracy: 0.7945 - loss: 0.4221
Epoch 4/10
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.8030 - loss: 0.4102
Epoch 5/10
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 801us/step - accuracy: 0.8204 - loss: 0.4002
Epoch 6/10
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 807us/step - accuracy: 0.8316 - loss: 0.3914
Epoch 7/10
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 961us/step - accuracy: 0.8419 - loss: 0.3837
Epoch 8/10
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 821us/step - accuracy: 0.8460 - loss: 0.3781
Epoch 9/10
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.8510 - loss: 0.3724  
Epoch 10/10
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 897us/step - accuracy: 0.8515 - loss: 0.3680


In [15]:
y_pred = model.predict(X_test)

y_pred = (y_pred > 0.5).astype(int)

print("Accuracy:", accuracy_score(y_test, y_pred))

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step  
Accuracy: 0.854
Confusion Matrix:
[[1546   61]
 [ 231  162]]


In [17]:

# Take customer details
print("Enter Customer Details:")

credit = float(input("Credit Score: "))
geography = input("Geography (France/Germany/Spain): ")
gender = input("Gender (Male/Female): ")
age = int(input("Age: "))
tenure = int(input("Tenure: "))
balance = float(input("Balance: "))
products = int(input("Number of Products: "))
card = int(input("Has Credit Card? (1=Yes, 0=No): "))
active = int(input("Is Active Member? (1=Yes, 0=No): "))
salary = float(input("Estimated Salary: "))

# Create customer data
customer = pd.DataFrame([{
    'CreditScore': credit,
    'Gender': 1 if gender.lower() == 'male' else 0,
    'Age': age,
    'Tenure': tenure,
    'Balance': balance,
    'NumOfProducts': products,
    'HasCrCard': card,
    'IsActiveMember': active,
    'EstimatedSalary': salary,
    'Geography_Germany': 1 if geography.lower() == 'germany' else 0,
    'Geography_Spain': 1 if geography.lower() == 'spain' else 0
}])

# Normalize customer data
customer = sc.transform(customer)

# Predict
prediction = model.predict(customer)

print("\nCustomer Prediction:")

if prediction[0][0] > 0.5:
    print("Customer WILL LEAVE the bank")
else:
    print("Customer WILL STAY with the bank")


Enter Customer Details:


Credit Score:  7
Geography (France/Germany/Spain):  spain
Gender (Male/Female):  male
Age:  32
Tenure:  30000
Balance:  25000
Number of Products:  4
Has Credit Card? (1=Yes, 0=No):  1
Is Active Member? (1=Yes, 0=No):  1
Estimated Salary:  32000


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step

Customer Prediction:
Customer WILL STAY with the bank
